# Analysis

In [173]:
import pandas as pd

race_results = pd.read_csv("data/raw/f1db-races-race-results.csv")
races = pd.read_csv("data/raw/f1db-races.csv")
quali_results = pd.read_csv("data/raw/f1db-races-qualifying-results.csv")
circuits = pd.read_csv("data/raw/f1db-circuits.csv")


races = races[races["year"].between(2010, 2026)]
race_results = race_results[race_results["year"].between(2010, 2026)]
quali_results = quali_results[quali_results["year"].between(2010, 2026)]

# One row per race
# Subset focuses only on the id -> race. Sum then adds up all the True values from duplicated() which only occur if an id is there for more than one time
print("Duplicate races:", races.duplicated(subset=["id"]).sum())

# One row per driver and race
print("Duplicate race results:", race_results.duplicated(subset=["raceId", "driverId"]).sum())

print("Duplicate qualifying results:", quali_results.duplicated(subset=["raceId", "driverId"]).sum())



Duplicate races: 0
Duplicate race results: 0
Duplicate qualifying results: 0


/var/folders/y6/yvr9h5p110qc_zbybx4w5p680000gn/T/ipykernel_34968/2600749808.py:3: DtypeWarning: Columns (31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  race_results = pd.read_csv("data/raw/f1db-races-race-results.csv")


- Races: Checks whether a race ID appears more than once.
- Race results: Checks whether a driver appears more than once in the same race.
- Qualifying results: Checks whether a driver appears more than once in the same qualifying session.

Especially important for new data updates

## Merge data sets

In [174]:
# how="inner" means that only for available ids of race_results, the values of race get joined (not too important here since the data is already very cleanly prepared)
# validate: raceId could be contained multiple times but Id must only appear once, else a MergeError would be triggered
# drop: drops id since we have raceId and id which are the same

model_data = race_results.merge(
    races[[
        "id",
        "date",
        "circuitId",
        "circuitType",
        "courseLength",
        "turns",
        "scheduledLaps"]],
    left_on="raceId",
    right_on="id",
    how="inner",
    validate="many_to_one"
).drop(columns="id")


# on: assigns to every driver the qualifying variables to the according race
# left: all the data of model_data stays, where the suiting parts from quali results get added
# rename is needed, since positionNumber is in the model_data used for the final position

model_data = model_data.merge(
    quali_results[[
        "raceId",
        "driverId",
        "q1Millis",
        "q2Millis",
        "q3Millis",
    ]],
    on=["raceId", "driverId"],
    how="left",
    validate="one_to_one"
)


## How to deal with missing values in gridPositionNumber

In [175]:
print(model_data["gridPositionText"].unique())

gridPositionNumber_nan = model_data[model_data["gridPositionText"].isna()]

['3' '2' '4' '1' '5' '7' '8' '6' '12' '11' '9' '10' '18' '13' '21' '15'
 '20' '14' 'PL' '19' '17' '16' '22' '24' '23' nan]


All the nan values are because of DNS or DNQ, thus they should be excluded. PL stands for "pit lane", meaning that the drivers had to start from the pit lane, which is more or less euqivalent as starting from the last position. Since the field hasn't always the same size, it can't be just set to 20.

In [176]:
model_data = model_data[model_data["gridPositionText"].notna()]

# transform("size"): size counts rows per group, where raceId is treated as the group

model_data["fieldSize"] = model_data.groupby("raceId")["driverId"].transform("size")


# .loc searches for the column gridPositionText with value "PL" and takes the related gridPositionNumber

model_data.loc[ model_data["gridPositionText"] == "PL", "gridPositionNumber"] = model_data["fieldSize"]

## Exlude and add features

In [177]:
columns_needed = [
    "raceId",
    "fieldSize",
    "year",
    "round",
    "date",
    "driverId",
    "constructorId",
    "circuitId",
    "circuitType",
    "courseLength",
    "turns",
    "scheduledLaps",
    "gridPositionNumber",
    "gridPositionText",
    "qualificationPositionNumber",
    "q1Millis",
    "q2Millis",
    "q3Millis",
    "positionDisplayOrder",
    "positionText",
    "reasonRetired"
]


model_data = model_data[columns_needed].copy()

In [178]:
model_data["qualifying_fraction"] = (
    (model_data["qualificationPositionNumber"] - 1)
    / (model_data["fieldSize"] - 1)
)

model_data["grid_fraction"] = (
    (model_data["gridPositionNumber"] - 1)
    / (model_data["fieldSize"] - 1)
)

model_data["finish_fraction"] = (
    (model_data["positionDisplayOrder"] - 1)
    / (model_data["fieldSize"] - 1)
)

Normalizing positions makes races with different field sizes comparable: finishing 20th out of 20 and 20th out of 22.

#### 5 race average

In [179]:
# transform: applies a computation on every group and returns its results back to the original row.
# shift(1): shifts all values one rows downwards, such that the result only gets included in the next races last five average
# rolling(5): uses five consecutive values
# min_periods says that the value should also be computed if only 4 instead of 5 values are available

model_data["driver_finish_5"] = (
    model_data.groupby("driverId")["finish_fraction"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)


#### 5 quali average

In [180]:
model_data["driver_quali_5"] = (
    model_data.groupby("driverId")["qualifying_fraction"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

#### Quali deviation feature

In [181]:
model_data["quali_vs_recent"] = (
    model_data["qualifying_fraction"]
    - model_data["driver_quali_5"]
)

#### Failure rate

In [182]:
model_data["failure"] = (model_data["positionText"].eq("DNF") | model_data["reasonRetired"].notna())

# .eq("DNF") is the same as == "DNF"
# .notna() gives back True, if a value isn't missing. (Everything that is missing in reasonRetiered hasn't retiered -> should return False)

model_data["failure_rate"] = (model_data.groupby("driverId")["failure"]
                                .transform(lambda failrate: failrate.shift(1).rolling(10, min_periods=5).mean()))

# mean values True and False as 0 and 1 leading to e.g. 0.4 -> 40%

#### Adding other average historical features

In [183]:
for window in [3, 5, 10, 20]:
    for source, name in [("finish_fraction", "finish"), ("qualifying_fraction", "quali"), ("failure", "failure_rate")]:
        model_data[f"driver_{name}_{window}"] = (model_data.groupby("driverId")[source]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean()))

This adds the average values not of the features computed above also for other race windows. i.e. 3, 5, 10 and 20 races

#### Standard deviations: consistency in races

In [184]:
for window in [5, 10]:
    model_data[f"driver_std_{window}"] = (
        model_data.groupby("driverId")["finish_fraction"].transform(lambda x: x.shift(1).rolling(window, min_periods=2).std())
    )

#### Several features for driver experience

Total experience in races

In [185]:
model_data["driver_experience"] = (
    model_data.groupby("driverId").cumcount()       # cumcount() counts the total apperances up to this point
)

Experience on this exact circuit (Possibly important if a new one gets intorduced)

In [186]:
model_data["driver_circuit_experience"] = (
    model_data.groupby(["driverId", "circuitId"]).cumcount()
)

How good a driver performs on a circuit. Once again only taking the previous 3 results into account, with a minimum of 1 race. If never raced this circuit before the drivers get a NaN, which later must be changed. Maybe to the median of the whold field or just ignored.

In [187]:
model_data["driver_circuit_form"] = (
    model_data.groupby(["driverId", "circuitId"])["finish_fraction"]
    .transform(lambda x:
                x.shift(1).rolling(3, min_periods=1).mean())
)

#### Team Performance

We create a separate team table with one row per team per race, so rolling averages count races rather than individual drivers. Later these data sets get merged together

In [188]:
team_data = (
    model_data.groupby(
        ["raceId", "constructorId", "circuitId"],
        as_index=False                          # to ensure that the correct indizes remain, which is important for later merges
    )
    .agg(                                       # compresses each constructor per race into a row and creates the columns
        team_finish=("finish_fraction", "mean"),        # new column name = used variables, method
        team_quali=("qualifying_fraction", "mean"),
        team_dnf=("failure", "mean")
    )
)

Now done for different windows in history. Once again only using previous values (shift(1))

In [189]:
for window in [3, 5, 10, 20]:
    for source in ["team_finish", "team_quali", "team_dnf"]:
        team_data[f"{source}_{window}"] = (
            team_data.groupby("constructorId")[source]
            .transform(lambda x: 
                       x.shift(1).rolling(window, min_periods=1).mean()))

Now the average team form on a certain circuit is added to the team_data table

In [190]:
team_data["team_circuit_form"] = (
    team_data.groupby(["constructorId", "circuitId"])["team_finish"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

Finally we merge these features to the original data set.

In [191]:
# create a list via list comprehension with all the names of the team features
team_features = [f"{name}_{window}" for window in [3, 5, 10, 20] for name in ["team_finish", "team_quali", "team_dnf"]] + ["team_circuit_form"]


model_data = model_data.merge(
    team_data[["raceId", "constructorId"] + team_features],
    on=["raceId", "constructorId"],
    how="left",
    validate="many_to_one"
)

#### Differences

Between Driver and Teammate

In [192]:
model_data["driver_team_difference"] = (
    model_data["driver_finish_5"] - model_data["team_finish_5"]
)

Team Qualifying vs recent Team performance

In [193]:
model_data["team_quali_vs_recent"] = (
    model_data.groupby(["raceId", "constructorId"])["qualifying_fraction"].transform("mean") - model_data["team_quali_5"]
)

#### Qualifying Features

Gap to the leader in the session in %. If already droped out, a NaN will be created!

In [194]:
for stage in [1, 2, 3]:
    column = f"q{stage}Millis"

    best_time = model_data.groupby("raceId")[column].transform("min")       # min is here the minimum, not minute as one might think because of time feature

    model_data[f"q{stage}_gap"] = (model_data[column] / best_time - 1) * 100

#### Create a factor variable for street tracks

In [195]:
model_data["street"] = (
    model_data["circuitType"].eq("STREET").astype(int)          # checks if the street type is equal to STREET. If so it assigns the variable 1
)

Create different starting and qualifying variables for street tracks. This could be particularly important because street tracks are known for their narrowness, which makes it even more difficult to overtake. With those additional variables the model can account for that. If it isn't a street circuit the variable is equal to 0 which automatically eliminates it out of the model. Thus, it works like a weighted factor variable.

In [196]:
model_data["grid_street"] = (
    model_data["grid_fraction"] * model_data["street"]
)

model_data["qualifying_street"] = (
    model_data["qualifying_fraction"] * model_data["street"]
)

#### Pit-Lane Start
The model should be able to give a start from the pit lane a different weight.

In [197]:
model_data["pit_lane"] = model_data["gridPositionText"].eq("PL").astype(int)

#### Failure Rates on circuit
Again we create a new data set to use rolling for the last 5 races, not driver performances

In [198]:
circuit_data = (
    model_data.groupby(["raceId", "circuitId"], as_index=False)
    .agg(circuit_dnf=("failure", "mean"))                           # create the variable circuit_dnf which is the mean failure rate from the failure feature
)


# use the last 5 races on this circuit to create the rolling average
circuit_data["circuit_dnf_history"] = (
    circuit_data.groupby("circuitId")["circuit_dnf"]
    .transform(lambda x:
                x.shift(1).rolling(5, min_periods=1).mean())
)


# merge the data into the original data set
model_data = model_data.merge(
    circuit_data[["raceId", "circuit_dnf_history"]],
    on="raceId",
    how="left",
    validate="many_to_one"
)

## Model feature selection

#### Categorical features

In [199]:
categorical_features = [
    "driverId",
    "constructorId",
    "circuitId",
    "circuitType"
]

#### Numerical features

In [ ]:
numeric_features = [
    "gridPositionNumber",
    "qualificationPositionNumber",
    "grid_fraction",
    "qualifying_fraction",
    "grid_change",
    "pit_lane",
    "street",
    "grid_street",
    "qualifying_street",
    "courseLength",
    "turns",
    "scheduledLaps",
    "q1_gap",
    # "q2_gap",
    # "q3_gap",
    "qualifying_vs_team",
    "driver_std_5",
    "driver_std_10",
    "driver_experience",
    "driver_circuit_form",
    "driver_circuit_experience",
    "driver_ewm",
    "team_circuit_form",
    "circuit_dnf_history",
    "driver_team_difference",
    "quali_vs_recent",
    "team_quali_vs_recent"
]

rolling_window_features = ["driver_finish","driver_quali","driver_dnf","team_finish","team_quali","team_dnf"]

for window in [3, 5, 10, 20]:
    for name in rolling_window_features:
        numeric_features.append(f"{name}_{window}")

q2_gap and q3_gap are left out because their missing values often indicate elimination in an earlier qualifying stage, making median imputation potentially misleading.

## Split Data for CV and Final Test

In [201]:
# This data is used to find the best model via CV
development_data = model_data[
    model_data["year"].between(2010, 2024)].copy()



# This data is for the final test
test_data = model_data[
    model_data["year"] == 2025].copy()

#### CV rolling window for 5 years

In [202]:
cv_splits = []

for validation_year in [2022, 2023, 2024]:
    train = development_data[
        development_data["year"].between(validation_year - 5, validation_year - 1)].copy()

    valid = development_data[
        development_data["year"] == validation_year].copy()

    cv_splits.append({
        "validation_year": validation_year,
        "train": train,
        "valid": valid,})

## Model Processing

## 